In [ ]:
!pip install seaborn
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import seaborn as sns
from PIL import Image
from collections import Counter
sns.set(style="whitegrid")

In [ ]:
train_df = pd.read_csv("/data/split/train.csv")
val_df = pd.read_csv("/data/split/val.csv")
train_val_df = pd.concat([train_df, val_df], ignore_index=True)
DATASET_CSV = "/data/prepared_dataset.csv"
df = pd.read_csv(DATASET_CSV)

print(f"Train+Val total images: {len(train_val_df)}")
train_val_df.head()

consensus_df = pd.read_csv("/data/split/consensus_test.csv")
print(f"Consensus test images: {len(consensus_df)}")
consensus_df.head()

In [ ]:
print(f"Total images in the initial, but already prepocessed image list: {len(df)}")

valid_count = 0
invalid_count = 0
invalid_files = []

for idx, row in df.iterrows():
    img_path = row["image_path"]
    try:
        with Image.open(img_path) as img:
            img.verify()
        valid_count += 1
    except Exception as e:
        invalid_count += 1
        invalid_files.append(img_path)

print(f"    - Valid images: {valid_count}")
print(f"    - Invalid images: {invalid_count}")

status_counts = {
    "Valid images": valid_count,
    "Invalid / unreadable images": invalid_count
}

status_df = pd.DataFrame(
    list(status_counts.items()),
    columns=["Status", "Count"]
)

plt.figure(figsize=(5, 5))
plt.pie(
    status_df["Count"],
    labels=status_df["Status"],
    autopct="%.1f%%",
    startangle=90
)
plt.title("Image validity in the initial dataset")
plt.axis("equal")
plt.show()


In [ ]:
def print_stats(df, name):
    print(f"=== {name} ===")
    print(f"Total images: {len(df)}")
    if 'label' in df.columns:
        print("Label counts:")
        print(df['label'].value_counts())
    print("Unique participants:", df['participant'].nunique() if 'participant' in df.columns else "N/A")
    print()

print_stats(train_val_df, "Train + Val")
print_stats(consensus_df, "Consensus Test")


In [ ]:
def show_sample_images(df, n=10):
    fig, axes = plt.subplots(1, n, figsize=(20,4))
    for i, ax in enumerate(axes):
        img = Image.open(df.iloc[i]['image_path'])
        ax.imshow(img)
        ax.set_title(df.iloc[i].get('label', ''))
        ax.axis('off')
    plt.show()

show_sample_images(train_val_df)
show_sample_images(consensus_df)


In [ ]:
def plot_image_sizes(df):
    sizes = [Image.open(p).size for p in df['image_path']]
    widths, heights = zip(*sizes)
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    sns.histplot(widths)
    plt.title("Widths")
    plt.subplot(1,2,2)
    sns.histplot(heights)
    plt.title("Heights")
    plt.show()

plot_image_sizes(train_val_df)
plot_image_sizes(consensus_df)
